# Day 10 — Pydantic v2: Advanced Validation

**Phase 1 · Module 1: Capstone** · ~30 min

Yesterday you made calls run in parallel. Today you make your **data safe** before it ever reaches your agent. A Pydantic model is a *bouncer at the door*: bad data gets stopped and told exactly why.

### What I practice today
- [ ] A plain model checks **types** — but not **business rules**
- [ ] `@field_validator` — check **one field** (an invoice amount must be **> 0**)
- [ ] Clean up a field before storing it (trim + upper-case a currency code)
- [ ] `@model_validator` — check **two fields together** (net + tax must equal total)
- [ ] `@computed_field` — **derive** a value (total-with-tax) instead of trusting the input
- [ ] See a `ValidationError` collect **all** the problems at once
- [ ] Bridge: this is the **validator node** in today's LangGraph Invoice-Triage capstone

> **No key / no install extras** — pure `pydantic` v2, runs fully offline.

## 1. A plain model checks *types*, not *rules*

Pydantic's first job is easy: coerce and type-check. Give it a model and it guarantees `amount` is a number, `currency` is a string, and so on. That already catches a lot — but it does **not** know that a bank invoice can't have a **negative** amount, or a **blank** vendor. Those are *business rules*, and the type system alone won't stop them.

In [4]:
from pydantic import BaseModel

class InvoiceBasic(BaseModel):
    vendor: str
    amount: float
    currency: str

# Types are fine, so Pydantic happily accepts nonsense:
bad = InvoiceBasic(vendor="", amount=-500.0, currency="pounds sterling")
print(bad)

vendor='' amount=-500.0 currency='pounds sterling'


☝ Every field is the *right type*, so the model says yes — even though the vendor is **empty**, the amount is **negative**, and the currency isn't a 3-letter code. Type-correct ≠ sensible. We need to teach the model our rules.

## 2. `@field_validator` — one field, one rule

A `@field_validator` is a function that runs on **one field** after Pydantic has type-checked it. Return the (possibly cleaned) value to accept it, or `raise ValueError(...)` to reject it. The plan's rule: **an invoice amount must be greater than 0.**

In [5]:
from pydantic import BaseModel, field_validator

class Invoice(BaseModel):
    vendor: str
    amount: float

    @field_validator("amount")
    @classmethod
    def amount_must_be_positive(cls, v: float) -> float:
        if v <= 0:
            raise ValueError("amount must be greater than 0")
        return v          # accepted — hand the value back

ok = Invoice(vendor="AWS", amount=420.50)
print("accepted:", ok)

try:
    Invoice(vendor="AWS", amount=-5)
except ValueError as e:
    print("rejected:", e)

accepted: vendor='AWS' amount=420.5
rejected: 1 validation error for Invoice
amount
  Value error, amount must be greater than 0 [type=value_error, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


☝ The good invoice passes; the negative one is stopped with **our own message**. Two things to notice: the validator is a `@classmethod`, and it must **return the value** on success (forgetting to return turns your field into `None`). One field, one clear rule.

## 3. A validator can also **clean** the value

Validators aren't only for rejecting — they can **normalise**. Currency codes should be a tidy 3-letter upper-case string (`GBP`), but people type `" gbp "`, `"Gbp"`, etc. Instead of rejecting, we trim and upper-case, then check it's valid. The stored value is always clean.

In [6]:
class Invoice(BaseModel):
    vendor: str
    amount: float
    currency: str

    @field_validator("amount")
    @classmethod
    def amount_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError("amount must be greater than 0")
        return v

    @field_validator("currency")
    @classmethod
    def normalise_currency(cls, v: str) -> str:
        v = v.strip().upper()               # clean it up
        allowed = {"GBP", "USD", "EUR"}
        if v not in allowed:
            raise ValueError(f"currency must be one of {allowed}")
        return v                            # store the tidy version

inv = Invoice(vendor="AWS", amount=420.50, currency="  gbp ")
print("stored currency ->", repr(inv.currency))   # 'GBP', trimmed + upper

stored currency -> 'GBP'


☝ Messy input `'  gbp '` comes out as a clean `'GBP'`. This is the quiet superpower of field validators: everything **downstream** of the model can trust the data is already normalised. No more `.strip().upper()` scattered across your codebase.

## 4. `@model_validator` — check two fields **together**

Some rules involve **more than one field**, so no single-field validator can see enough. Example: an invoice has a `net`, a `tax`, and a `total`, and they must add up. A `@model_validator(mode="after")` runs **once, after all fields are set**, so it can compare them. `self` is the finished object.

In [7]:
from pydantic import model_validator

class LineItem(BaseModel):
    net: float
    tax: float
    total: float

    @model_validator(mode="after")
    def totals_must_add_up(self):
        expected = round(self.net + self.tax, 2)
        if round(self.total, 2) != expected:
            raise ValueError(f"total {self.total} != net+tax {expected}")
        return self          # after-validators return self

good = LineItem(net=100.0, tax=20.0, total=120.0)
print("accepted:", good)

try:
    LineItem(net=100.0, tax=20.0, total=999.0)   # doesn't add up
except ValueError as e:
    print("rejected:", e)

accepted: net=100.0 tax=20.0 total=120.0
rejected: 1 validation error for LineItem
  Value error, total 999.0 != net+tax 120.0 [type=value_error, input_value={'net': 100.0, 'tax': 20.0, 'total': 999.0}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


☝ A single-field validator could never catch this — the rule is a *relationship between fields*. `mode="after"` means it runs on the fully-built object, so `self.net`, `self.tax`, `self.total` are all available. Return `self` to accept. Use this for any cross-field rule: *due date after issue date*, *discount ≤ subtotal*, and so on.

## 5. `@computed_field` — derive it, don't trust it

If a value can be **calculated** from other fields, don't accept it as input — **compute** it. A `@computed_field` is a read-only property that shows up in the output (and in `model_dump()`), so callers see it, but nobody can pass a wrong one in. Here: `total_with_tax` from `subtotal` at a fixed 20% VAT.

In [8]:
from pydantic import computed_field

class Invoice(BaseModel):
    vendor: str
    subtotal: float

    @field_validator("subtotal")
    @classmethod
    def positive(cls, v):
        if v <= 0:
            raise ValueError("subtotal must be > 0")
        return v

    @computed_field                     # appears in output, can't be set
    @property
    def total_with_tax(self) -> float:
        return round(self.subtotal * 1.20, 2)   # +20% VAT

inv = Invoice(vendor="AWS", subtotal=100.0)
print("total_with_tax:", inv.total_with_tax)
print("model_dump  ->", inv.model_dump())

total_with_tax: 120.0
model_dump  -> {'vendor': 'AWS', 'subtotal': 100.0, 'total_with_tax': 120.0}


☝ You only supplied `subtotal`; `total_with_tax` appeared automatically and shows up in `model_dump()`. Because it's derived, it's **always consistent** — there's no way to send `subtotal=100` with a mismatched total. One source of truth.

## 6. Errors come back **all at once**, machine-readable

When several things are wrong, Pydantic doesn't stop at the first — it collects **every** error and raises one `ValidationError`. `.errors()` gives a structured list (field location + message) you can log, return as an API 422, or show a user. Great for forms and for an agent's validator node.

In [9]:
from pydantic import ValidationError

class Invoice(BaseModel):
    vendor: str
    amount: float
    currency: str

    @field_validator("vendor")
    @classmethod
    def vendor_not_blank(cls, v):
        if not v.strip():
            raise ValueError("vendor must not be blank")
        return v.strip()

    @field_validator("amount")
    @classmethod
    def positive(cls, v):
        if v <= 0:
            raise ValueError("amount must be > 0")
        return v

    @field_validator("currency")
    @classmethod
    def known_ccy(cls, v):
        if v.upper() not in {"GBP", "USD", "EUR"}:
            raise ValueError("unknown currency")
        return v.upper()

try:
    Invoice(vendor="  ", amount=-1, currency="XYZ")   # three things wrong
except ValidationError as e:
    print(f"{e.error_count()} problems found:")
    for err in e.errors():
        print(f"  - {err['loc'][0]}: {err['msg']}")

3 problems found:
  - vendor: Value error, vendor must not be blank
  - amount: Value error, amount must be > 0
  - currency: Value error, unknown currency


☝ Three bad fields → **three** errors in one shot, each tagged with the field name. That structured list is exactly what a service returns as a `422` response, or what an agent writes into its state so a later node (or a human) can act on every issue at once — not one-by-one.

## 7. Tie-in — this is the **validator node** in today's capstone

In today's LangGraph notebook you build an **Invoice-Triage Agent**: `OCR → classify → validate → human-gate → save`. The **validate** step is a Pydantic model just like §6. If it passes, the invoice flows to saving; if `ValidationError` fires, the graph **routes to a human** instead of writing bad data to the database.

Same three tools, one job each:

In [10]:
# a compact preview of the capstone's validator model
class TriageInvoice(BaseModel):
    vendor: str
    amount: float
    currency: str

    @field_validator("amount")
    @classmethod
    def _pos(cls, v):
        if v <= 0:
            raise ValueError("amount must be > 0")
        return v

    @computed_field
    @property
    def total_with_tax(self) -> float:
        return round(self.amount * 1.20, 2)

def validate_node(parsed: dict) -> dict:
    "Return a state update: valid + errors — exactly what a LangGraph node does."
    try:
        inv = TriageInvoice(**parsed)
        return {"valid": True, "errors": [], "total": inv.total_with_tax}
    except ValidationError as e:
        return {"valid": False, "errors": [x["msg"] for x in e.errors()]}

print(validate_node({"vendor": "AWS", "amount": 420.5, "currency": "GBP"}))
print(validate_node({"vendor": "AWS", "amount": -5,    "currency": "GBP"}))

{'valid': True, 'errors': [], 'total': 504.6}
{'valid': False, 'errors': ['Value error, amount must be > 0']}


☝ `validate_node` swallows the exception and returns a **state update** (`valid`, `errors`) instead of crashing — that's the shape every LangGraph node uses. Tomorrow's graph reads `valid` to decide: **save** it, or **send it to a human**. Your Pydantic model is the gatekeeper.

## 8. Recap + checklist

| Tool | What it does | Rule of thumb |
|------|--------------|---------------|
| **plain `BaseModel`** | checks **types**, coerces | free, but doesn't know your business rules |
| **`@field_validator`** | one field: reject **or clean** | return the value (cleaned) or `raise ValueError` |
| **`@model_validator(mode="after")`** | compares **fields together** | for cross-field rules; return `self` |
| **`@computed_field`** | **derives** a value | don't accept what you can calculate |
| **`ValidationError.errors()`** | **all** problems, structured | log it / return 422 / write to agent state |

**Takeaway:** a Pydantic model is a bouncer — it type-checks, enforces your rules, cleans messy input, and derives what it can. Put one at every boundary (API input, LLM output, agent state) and everything downstream can **trust** its data. That's the **validator node** you wire up in today's LangGraph capstone.